## HEHE KITA MAIN DATA SEK

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
plt.style.use('ggplot')
import random
import numpy as np

from warnings import filterwarnings
filterwarnings("ignore")

### CREATE USER DATA

In [ ]:
users_data = []
number_users = 5000

for i in range(number_users):
    monthly_income = np.random.lognormal(mean=14, sigma=0.7)
    credit_score = np.random.normal(loc=0.6, scale=0.15)
    credit_score = np.clip(credit_score, 0, 1)
    
    users_data.append({
        "name": f"name-{i+1}",
        "age": random.randint(18, 45),
        "occupation": random.randint(0, 10),
        "monthly_income": int(monthly_income),
        "social_status": int(np.random.exponential(scale=1.2)),
        "location": random.randint(0, 5),
        "credit_score": credit_score,
    })

In [ ]:
df = pd.DataFrame(users_data)
df

In [ ]:
df.hist(column='age', bins=25)
plt.show()

In [ ]:
df.hist(column='occupation', bins=4)
plt.show()

In [ ]:
df.hist(column='monthly_income', bins=100)
plt.show()

In [ ]:
df.hist(column='location', bins=4)
plt.show()

In [ ]:
df.hist(column='social_status', bins=100)
plt.show()

In [ ]:
df.hist(column='credit_score', bins=100)
plt.show()

### CREATE USER HUB DATA BY HOW THEY RELATED EACH OTHER

In [ ]:
user_hub = []
existing_relations = set()

for i in range(len(users_data)):
    for j in range(i + 1, len(users_data)):
        user_1 = users_data[i]
        user_2 = users_data[j]

        if abs(user_1['location'] - user_2['location']) < 2:

            if random.random() < 0.05:

                pair = tuple(sorted([user_1['name'], user_2['name']]))

                if pair not in existing_relations:
                    existing_relations.add(pair)

                    user_hub.append({
                        "user_1": pair[0],
                        "user_2": pair[1],
                        "related_kind": random.randint(1, 5)
                    })

In [ ]:
pd.DataFrame(user_hub)

### CREATE USER ENGANGEMENT DATA BY HOW THE DO THEIR ACTIVITY

In [ ]:
import random
import numpy as np
from collections import defaultdict
from datetime import datetime, timedelta

# -----------------------------------
# 1. Build adjacency map
# -----------------------------------
relation_map = defaultdict(dict)

for rel in user_hub:
    u1 = rel["user_1"]
    u2 = rel["user_2"]
    related_level = rel["related_kind"]
    
    relation_map[u1][u2] = related_level
    relation_map[u2][u1] = related_level


# -----------------------------------
# 2. Generate Activity Records
# -----------------------------------
activity_records = []

start_time = datetime.now()

for i in range(100):

    base_user = random.choice(users_data)["name"]
    participants = {base_user}

    if base_user in relation_map:
        for other_user, related_level in relation_map[base_user].items():

            join_prob = 0.05 + (related_level * 0.08)

            if random.random() < join_prob:
                participants.add(other_user)

    participants = list(participants)

    base_value = np.random.normal(loc=5_000_000, scale=2_000_000)

    value_creation = max(0, base_value * (1 + 0.3 * (len(participants) - 1)))

    activity_records.append({
        "user_names": participants,
        "activity": random.randint(0, 20),
        "value_creation": int(value_creation),
        "time_stamps": start_time + timedelta(minutes=i)
    })

In [ ]:
activity_records_df = pd.DataFrame(activity_records)
activity_records_df

In [ ]:
activity_records_df.hist(column='activity', bins=20)

plt.show()

In [ ]:
activity_records_df.hist(column='value_creation', bins=25)

plt.show()

### SAVE DATA

In [ ]:
recorded_act = pd.DataFrame(activity_records)

recorded_act.to_csv("../datas/recorded-actifity.csv", index=False)

In [ ]:
users_hub = pd.DataFrame(user_hub)

users_hub.to_csv("../datas/user-relation.csv", index=False)

In [ ]:
users = pd.DataFrame(users_data)

users.to_csv("../datas/users.csv", index=False)

## Test Of using Data

In [ ]:
actifity_data = pd.read_csv("../datas/recorded-actifity.csv")
user_data = pd.read_csv("../datas/users.csv")

### User Data

In [ ]:
user_data

In [ ]:
user_data_feature = user_data.drop(columns=['name', 'credit_score'], axis=1)
user_data_feature

In [ ]:
print(f"max value of age is {user_data_feature['age'].max()} and min value of age is {user_data_feature['age'].min()}")

In [ ]:
print(f"max value of monthly income is {user_data_feature['monthly_income'].max()} and min value of monthly income is {user_data_feature['monthly_income'].min()}")

In [ ]:
print(f"max value of occupation is {user_data_feature['occupation'].max()} and min value of occupation is {user_data_feature['occupation'].min()}")

In [ ]:
print(f"max value of social_status is {user_data_feature['social_status'].max()} and min value of social_status is {user_data_feature['social_status'].min()}")

In [ ]:
print(f"max value of location is {user_data_feature['location'].max()} and min value of location is {user_data_feature['location'].min()}")

In [ ]:
user_data_feature.shape

### Activity Data

In [ ]:
actifity_data.head()

In [ ]:
print(f"max value of value_creation is {actifity_data['value_creation'].max()} and min value of value_creation is {actifity_data['value_creation'].min()}")

In [ ]:
print(f"max value of activity is {actifity_data['activity'].max()} and min value of activity is {actifity_data['activity'].min()}")

In [ ]:
adjency_dictionary = {
    "idx": [i for i in user_data['name'].to_list()]
}

for i in user_data['name']:
    adjency_dictionary[i] = [0] * len(user_data['name'].to_list())

In [ ]:
adjency_dictionary_dataframe = pd.DataFrame(adjency_dictionary)

adjency_dictionary_dataframe.set_index("idx", inplace=True)

adjency_dictionary_dataframe.columns

In [ ]:
from tqdm.notebook import tqdm

In [ ]:
import ast

max_value_creation_ever_recorded = 116612972
min_value_creation_ever_recorded = 4540178

for names, value_creation, act in tqdm(zip(
    actifity_data['user_names'],
    actifity_data['value_creation'],
    actifity_data['activity']
), total=len(actifity_data['user_names'].to_list()), desc="processing the data.."):
    names_as_litearal_list = ast.literal_eval(names)
    normalize_value_creation = (value_creation - min_value_creation_ever_recorded) / max_value_creation_ever_recorded
    normalize_actifity = act / 20
    adjency_vertice_score = (normalize_value_creation * 10 + normalize_actifity * 8) / 18
    
    for i in range(len(names_as_litearal_list)):
        for j in range(i+1, len(names_as_litearal_list) - 1 - i):
            name1 = names_as_litearal_list[i]
            name2 = names_as_litearal_list[j]
            adjency_dictionary_dataframe.loc[name1, name2] += adjency_vertice_score
            adjency_dictionary_dataframe.loc[name2, name1] += adjency_vertice_score

In [ ]:
adjency_dictionary_dataframe.info()

In [ ]:
adjency_dictionary_matrix = adjency_dictionary_dataframe.to_numpy()

In [ ]:
adjency_dictionary_matrix.shape

In [ ]:
adjency_dictionary_matrix.max()

## Data Tabulation

In [ ]:
class DataTabulate:
    def __init__(self):
        self.__nodes = None
        self.__adjency_matrix = None

## MODEL DEVELOPMENT

In [ ]:
class GCN_CREDIT_SCORING_AWARNESS:
    def __init__(self):
       pass

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
plt.style.use('ggplot')

matrix_images = np.array([
    [1, 2, 3, 4],
    [4, 4, 4, 4],
    [4, 3, 2, 1],
    [4, 4, 4, 4],
], dtype=np.float64)

kernel_matrix = np.array([
    [2, 1],
    [1, 2],
], dtype=np.float64)

In [ ]:
plt.imshow(matrix_images, vmin=1, vmax=4)
plt.show()

In [ ]:
plt.imshow(kernel_matrix, vmin=1, vmax=4)
plt.show()

In [ ]:
matrix_images

In [ ]:
kernel_matrix

In [ ]:
plt.imshow(conv_res)
plt.show()